In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

In [2]:
best_xgb = joblib.load("../models/best_xgboost.joblib")
encoder = joblib.load("../models/agegroup_encoder.joblib")

In [3]:
df = pd.read_csv("../data/cleaned/cleaned_credit_default.csv")

y = df["SeriousDlqin2yrs"]
X = df.drop(columns=["SeriousDlqin2yrs"])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

categorical_cols = ["AgeGroup"]
numeric_cols = X_train.columns.difference(categorical_cols).tolist()

X_train_cat = encoder.transform(X_train[categorical_cols])
X_test_cat = encoder.transform(X_test[categorical_cols])

X_train_num = X_train[numeric_cols].to_numpy()
X_test_num = X_test[numeric_cols].to_numpy()

X_train_final = np.hstack([X_train_num, X_train_cat])
X_test_final = np.hstack([X_test_num, X_test_cat])

In [4]:
y_prob_test = best_xgb.predict_proba(X_test_final)[:, 1]

print(y_prob_test[:10])

[0.48726138 0.03868934 0.04883744 0.30711648 0.0412397  0.04259463
 0.19779423 0.01932161 0.24359605 0.06456188]


In [5]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.05, 0.96, 0.05)

threshold_results = []

for threshold in thresholds:
    y_pred = (y_prob_test >= threshold).astype(int)

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

threshold_results = pd.DataFrame(threshold_results)

threshold_results

C:\Users\Rupali Waghmare\miniconda3\envs\AI_ML\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,Threshold,Precision,Recall,F1
0,0.05,0.113939,0.941236,0.203271
1,0.10,0.156752,0.874367,0.265845
2,0.15,0.191692,0.806484,0.309758
3,0.20,0.228919,0.733029,0.348885
4,0.25,0.268928,0.658561,0.381904
5,0.30,0.307869,0.610436,0.409307
6,0.35,0.344328,0.565856,0.428133
7,0.40,0.367048,0.520263,0.430427
8,0.45,0.394507,0.480243,0.433173
9,0.50,0.422964,0.436677,0.429711


In [6]:
best_threshold_row = threshold_results.loc[
    threshold_results["F1"].idxmax()
]

best_threshold_row

Threshold    0.450000
Precision    0.394507
Recall       0.480243
F1           0.433173
Name: 8, dtype: float64

In [7]:
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train_final,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print("Training:", X_train_sub.shape)
print("Validation:", X_val.shape)

Training: (95369, 24)
Validation: (23843, 24)


In [8]:
from xgboost import XGBClassifier

xgb_validation = XGBClassifier(
    colsample_bytree=0.8,
    learning_rate=0.05,
    max_depth=3,
    n_estimators=200,
    scale_pos_weight=3,
    subsample=0.8,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

xgb_validation.fit(X_train_sub, y_train_sub)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'logloss'


In [9]:
y_prob_val = xgb_validation.predict_proba(X_val)[:, 1]

In [10]:
thresholds = np.arange(0.05, 0.96, 0.05)

validation_results = []

for threshold in thresholds:
    y_pred_val = (y_prob_val >= threshold).astype(int)

    precision = precision_score(y_val, y_pred_val, zero_division=0)
    recall = recall_score(y_val, y_pred_val, zero_division=0)
    f1 = f1_score(y_val, y_pred_val, zero_division=0)

    validation_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

validation_results = pd.DataFrame(validation_results)

validation_results

,Threshold,Precision,Recall,F1
0,0.05,0.113569,0.939241,0.202635
1,0.10,0.155481,0.875316,0.264057
2,0.15,0.190891,0.803797,0.308515
3,0.20,0.227950,0.736076,0.348099
4,0.25,0.271800,0.674684,0.387495
5,0.30,0.308534,0.624684,0.413057
6,0.35,0.347470,0.591139,0.437676
7,0.40,0.379130,0.551899,0.449485
8,0.45,0.413244,0.509494,0.456349
9,0.50,0.447500,0.453165,0.450314


In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

final_threshold = 0.45

y_pred_final = (
        y_prob_test >= final_threshold
).astype(int)

print("Threshold :", final_threshold)
print("Accuracy  :", accuracy_score(y_test, y_pred_final))
print("Precision :", precision_score(y_test, y_pred_final))
print("Recall    :", recall_score(y_test, y_pred_final))
print("F1        :", f1_score(y_test, y_pred_final))
print("ROC-AUC   :", roc_auc_score(y_test, y_prob_test))

Threshold : 0.45
Accuracy  : 0.9167561401154207
Precision : 0.3945068664169788
Recall    : 0.48024316109422494
F1        : 0.433173406442769
ROC-AUC   : 0.8621544414433995


In [12]:
cm = confusion_matrix(y_test, y_pred_final)

print(cm)

[[26375  1455]
 [ 1026   948]]


# Cost Based Threshold (To ensure less False Negative)

In [13]:
from sklearn.metrics import confusion_matrix

In [18]:
# Business Cost
C_FN = 5   # Cost of False Negative (Missing an Defaulter)
C_FP = 1    # Cost of False Positive (Classifying Non defaulter as Defaulter)

thresholds = np.arange(0.05 , 0.96 , 0.01)

results = []

for threshold in thresholds:

    y_pred = (y_prob_test >= threshold).astype(int)

    tn , fp , fn , tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()

    cost = C_FN * fn + C_FP * fp

    results.append({
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Cost": cost
    })

    cost_df = pd.DataFrame(results)
    best_cost = cost_df.loc[cost_df["Cost"].idxmin()]

cost_df

,Threshold,TN,FP,FN,TP,Cost
0,0.05,13381,14449,116,1858,15029
1,0.06,14910,12920,140,1834,13620
2,0.07,16069,11761,174,1800,12631
3,0.08,17052,10778,188,1786,11718
4,0.09,17872,9958,214,1760,11028
...,...,...,...,...,...,...
86,0.91,27829,1,1963,11,9816
87,0.92,27830,0,1971,3,9855
88,0.93,27830,0,1973,1,9865
89,0.94,27830,0,1974,0,9870


## Conclusion

Cost-based threshold analysis shows that the optimal threshold depends on the relative cost of false negatives and false positives. With the selected cost ratio of **5:1**, where missing a defaulter is five times more costly than incorrectly flagging a non-defaulter, the minimum cost on the evaluated data occurs at a threshold of **0.36**, with a total cost of **6396**.

At the cost-optimal threshold of **0.36**, the model achieves a higher recall of approximately **0.5593**, meaning more actual defaulters are identified, but precision decreases to approximately **0.3505** because more non-defaulters are classified as high risk.

Compared with the F1-optimized threshold of **0.45**, the cost-based threshold prioritizes reducing false negatives rather than maximizing the balance between precision and recall. Therefore, **threshold selection should depend on the business objective and the relative cost of different types of errors**.

> **Note:** The threshold should ideally be selected using validation data and then evaluated on the unseen test set to avoid test-set information leakage.